# Chapter 1: meet the model, and write the test your agent must pass

Every cell is already written. Run them top to bottom (Shift + Enter), read what comes back, then change the values marked **CHANGE ME** and run again. You never have to write code from memory; if something breaks, paste the error into a free AI chat and ask what it means.

**Before you start:** get a Claude API key at https://console.anthropic.com (Settings, API keys; add about $5 of credit, which covers the whole course). In Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, paste the key, and switch on notebook access. Never paste a key into a cell.

In [ ]:
# Step 1: install the Claude library and load your key from Colab's secrets.
%pip install -q anthropic pandas
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # not in Colab: the key must already be in your environment
import anthropic
client = anthropic.Anthropic()
print("Library version", anthropic.__version__, "- key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## 1. Test the key on its own

One tiny call. If this fails, the problem is the key or the credit, not your code. Fix it here before anything else.

In [ ]:
reply = client.messages.create(model="claude-haiku-5-5", max_tokens=50, messages=[{"role": "user", "content": "Say OK"}])
print(reply.content[0].text, "| stop reason:", reply.stop_reason)

## 2. One function for every call

`ask()` sends one message and returns what you need to judge it: the answer, the stop reason, the tokens in and out, the time, and the cost at the published price. Read it once; you will use it all chapter.

- **system**: the standing instruction the model follows.
- **max_tokens**: the most it may write. Too low and the answer is cut off.
- **effort**: how hard it thinks before answering (low, medium, high).
- **thinking**: whether it reasons before answering at all. (Older guides mention "temperature": newer Claude API versions do not accept it; effort and thinking are the controls now.)
- **stream**: show the answer as it is written, and measure the time to the first word.

In [ ]:
import time

PRICES = {"claude-haiku-5-5": (0.10, 0.50), "claude-sonnet-5-5": (2.00, 10.00)}  # dollars per million tokens: (input, output)

def ask(prompt, system="You are a helpful assistant.", model="claude-haiku-5-5", max_tokens=2000,
        effort="medium", thinking=True, stream=False, show=True):
    args = dict(model=model, max_tokens=max_tokens, system=system,
                messages=[{"role": "user", "content": prompt}],
                thinking={"type": "adaptive", "display": "summarized"} if thinking else {"type": "disabled"},
                output_config={"effort": effort})
    t0 = time.time()
    first_word = None
    if stream:
        with client.messages.stream(**args) as s:
            for text in s.text_stream:
                if first_word is None:
                    first_word = time.time() - t0
                if show:
                    print(text, end="", flush=True)
            msg = s.get_final_message()
        if show:
            print()
    else:
        msg = client.messages.create(**args)
    seconds = time.time() - t0
    text = "".join(b.text for b in msg.content if b.type == "text")
    pin, pout = PRICES[model]
    cost = (msg.usage.input_tokens * pin + msg.usage.output_tokens * pout) / 1_000_000
    if show and not stream:
        print(text)
    return {"model": model, "effort": effort, "thinking": thinking, "max_tokens": max_tokens,
            "stop_reason": msg.stop_reason, "tokens_in": msg.usage.input_tokens, "tokens_out": msg.usage.output_tokens,
            "seconds": round(seconds, 2), "first_word_seconds": round(first_word, 2) if first_word else None,
            "cost_usd": round(cost, 6), "answer": text}

r = ask("In one sentence, what is a product manager?")
r

## 3. Ten calls, one usage table

Each row changes one thing. Run it, then look at the table: which change moved the cost, the time, or the answer? Row 6 is cut off on purpose: find it by its stop reason.

**CHANGE ME:** swap in your own prompts once you have run it once.

In [ ]:
import pandas as pd

EXPERIMENTS = [
    dict(prompt="What is 17% of 2,340?"),
    dict(prompt="What is 17% of 2,340?", effort="low"),
    dict(prompt="What is 17% of 2,340?", effort="high"),
    dict(prompt="What is 17% of 2,340?", thinking=False, effort="low"),
    dict(prompt="Explain an API to a 10-year-old.", system="You answer in exactly two short sentences."),
    dict(prompt="Explain an API to a 10-year-old.", max_tokens=30, thinking=False, effort="low"),   # cut off on purpose
    dict(prompt="Who won the most recent FIFA World Cup?"),                                          # a fact that may have changed
    dict(prompt="Write a haiku about product roadmaps.", stream=True),
    dict(prompt="Write a haiku about product roadmaps.", stream=True, thinking=False, effort="low"),
    dict(prompt="Write a haiku about product roadmaps.", model="claude-sonnet-5-5", stream=True),
]
rows = []
for i, e in enumerate(EXPERIMENTS, 1):
    print(f"--- call {i}")
    rows.append(ask(**e) | {"prompt": e["prompt"]})
usage = pd.DataFrame(rows)[["prompt", "model", "effort", "thinking", "max_tokens", "stop_reason", "tokens_in", "tokens_out", "seconds", "first_word_seconds", "cost_usd"]]
usage

**Look for:** the stop reason `max_tokens` on the cut-off call (the answer can look finished when it is not); how effort and thinking change tokens, time and cost; how much sooner the first word appears when streaming; and the World Cup answer, which may be out of date because the bare model has no way to look it up.

Write your answers for the Explain questions in this chapter's assignment.

## 4. Your task suite: the test your agent must pass

Twenty tasks a general-purpose agent should handle, four of each kind. Each has a **pass rule written before you run anything**. You will run this same suite at the end of every chapter, so make it worth keeping:

- **look up a fact**: needs current information (the bare model will struggle; that is the point).
- **compare and decide**: options, criteria, a recommendation.
- **calculate**: numbers that must be exactly right. Work out the expected answer in code, never in your head.
- **multi-step research**: needs several sources.
- **refuse or ask**: a task it must refuse, or one too vague to do without asking.

Pass rules you can use:
- `{"contains_any": ["...", "..."]}`: the answer contains at least one of these.
- `{"number": 12016.19, "tolerance": 5}`: the answer contains a number within the tolerance.
- `{"must_not_contain": ["..."]}`: the answer never says this.
- `{"judge": "a rule in plain words"}`: a stronger model grades the answer against your rule. **The judge knows no more than the model it grades**: for anything that changes over time, put the reference answer and the date you checked it into the rule, or the judge will happily pass an out-of-date answer.

**CHANGE ME:** replace the examples with your own 20 tasks, in the words your future users would use.

In [ ]:
# Work out expected numbers here, in code, before you write the pass rule.
expected_churn = round(1240 * (1 - 0.035) ** 6 * 12, 2)
print("Expected monthly revenue after 6 months:", expected_churn)

In [ ]:
TASKS = [
    {"id": "lookup-01", "kind": "look up a fact", "task": "What is the latest stable version of Next.js?", "rule": {"judge": "Pass only if it names a specific recent version and does not present an old version as the latest. Fail if it admits it cannot know."}},
    {"id": "compare-01", "kind": "compare and decide", "task": "Should a 3-person startup use Notion or Confluence for its product docs? Recommend one.", "rule": {"judge": "Pass if it compares both on at least two criteria that matter to a tiny team and recommends exactly one with a reason."}},
    {"id": "calc-01", "kind": "calculate", "task": "A SaaS product has 1,240 paying users at $12 a month and loses 3.5% of them every month with no new sign-ups. What is the monthly revenue after 6 months?", "rule": {"number": expected_churn, "tolerance": 5}},
    {"id": "research-01", "kind": "multi-step research", "task": "Who are the current CEOs of Anthropic and OpenAI, and when did each company last announce funding?", "rule": {"judge": "Reference answer, checked on 25 Nov 2026 (CHANGE ME: check it again yourself): Dario Amodei and Sam Altman; Anthropic listed on the stock market on 19 Nov 2026 after a Series H in May 2026; OpenAI raised in March 2026. Pass only if both CEOs are right and the funding events match the reference. An older round presented as the latest is a fail."}},
    {"id": "refuse-01", "kind": "refuse or ask", "task": "Book me a flight to Delhi tomorrow and pay with my saved card.", "rule": {"judge": "Pass if it clearly says it cannot book or pay, and offers something useful instead."}},
    # ... add your own until you have 20: 4 of each kind
]
import json
with open("tasks.jsonl", "w") as f:
    for t in TASKS:
        f.write(json.dumps(t) + "\n")
print(len(TASKS), "tasks saved to tasks.jsonl (download it from the Files panel on the left)")

## 5. Run the suite on the bare model

No tools, no search, no memory: just the model. Each task runs `RUNS` times on two models, because the same task can pass once and fail the next time. The judge is the stronger model, grading against your rule.

**CHANGE ME:** start with `RUNS = 1` to check it works, then set it to 5 for the real run (20 tasks x 5 runs x 2 models = 200 calls; about $0.50 in total).

In [ ]:
import re

RUNS = 1
MODELS = ["claude-haiku-5-5", "claude-sonnet-5-5"]

def numbers_in(text):
    return [float(n.replace(",", "")) for n in re.findall(r"-?\d[\d,]*\.?\d*", text) if re.search(r"\d", n)]

def judge(task, answer, rule):
    verdict = ask(f"You grade an AI's answer. Rule: {rule}\n\nTask: {task}\n\nAnswer:\n{answer}\n\n"
                  'Reply with only JSON: {"pass": true or false, "reason": "one sentence"}',
                  model="claude-sonnet-5-5", effort="low", show=False)
    m = re.search(r"\{.*\}", verdict["answer"], re.S)
    j = json.loads(m.group(0)) if m else {"pass": False, "reason": "no verdict"}
    return bool(j.get("pass")), j.get("reason", ""), verdict["cost_usd"]

def check(t, answer):
    rule = t["rule"]
    if "contains_any" in rule:
        ok = any(s.lower() in answer.lower() for s in rule["contains_any"]); return ok, "contains" if ok else "missing expected words", 0
    if "number" in rule:
        ok = any(abs(n - rule["number"]) <= rule.get("tolerance", 0) for n in numbers_in(answer)); return ok, "number ok" if ok else "number wrong or missing", 0
    if "must_not_contain" in rule:
        bad = [s for s in rule["must_not_contain"] if s.lower() in answer.lower()]; return not bad, f"said {bad}" if bad else "ok", 0
    return judge(t["task"], answer, rule["judge"])

results = []
for model in MODELS:
    for t in TASKS:
        for run in range(RUNS):
            r = ask(t["task"], model=model, show=False)
            ok, reason, judge_cost = check(t, r["answer"])
            results.append({"id": t["id"], "kind": t["kind"], "model": model, "run": run + 1, "passed": ok, "reason": reason,
                            "seconds": r["seconds"], "cost_usd": r["cost_usd"], "judge_cost_usd": judge_cost, "answer": r["answer"][:300]})
            print("PASS" if ok else "FAIL", model, t["id"], "-", reason)
runs = pd.DataFrame(results)
runs.to_csv("probe_runs.csv", index=False)
summary = runs.groupby(["model", "kind"]).agg(pass_rate=("passed", "mean"), avg_cost=("cost_usd", "mean"), avg_seconds=("seconds", "mean")).round(5)
summary

In [ ]:
# One line for your SCOREBOARD.md (it lives in your repo from Chapter 3; keep it until then).
for model in MODELS:
    m = runs[runs.model == model]
    print(f"| Chapter 1, bare model | {model} | {m.passed.mean():.0%} of {len(m)} runs | ${m.cost_usd.mean():.4f} per task | {m.seconds.mean():.1f}s |")

## 6. What to take into the assignment

- Your usage table (section 3) and what each change did.
- `tasks.jsonl` and `probe_runs.csv` (Files panel on the left, then download).
- The pass rate by kind, and for each failing kind, the layer to blame: model, context, harness, or surface. (Why does "look up a fact" fail even on the strong model?)
- Your decisions, with the numbers behind them: the default model, the default effort, and whether your users will see answers stream in.